In [59]:
import pandas as pd
import numpy as np

In [34]:
destination_df = pd.read_csv(
    "../data/cleaned/destination_features.csv"
)
print(destination_df.shape)
print(destination_df.head())



(92, 47)
        Destination          State   Category Best_Season  Average_Budget  \
0               Goa            Goa      Beach     Nov-Feb           30000   
1      Ladakh (Leh)         Ladakh   Mountain     Mar-Jun           35000   
2            Jaipur      Rajasthan   Heritage     Oct-Mar           22000   
3  Varanasi (Kashi)  Uttar Pradesh  Spiritual     Oct-Mar           18000   
4              Agra  Uttar Pradesh   Heritage     Oct-Mar           22000   

   Recommended_Trip_Duration   Nearest_Airport Popularity  Family_Friendly  \
0                          4       Goa Airport       High                1   
1                          5    Ladakh Airport       High                1   
2                          3    Jaipur Airport       High                1   
3                          2  Varanasi Airport       High                1   
4                          3      Agra Airport       High                1   

   Adventure_Score  ...  Average_Hotel_Price  Minimum_Hotel

In [35]:
# Display all destination names.
#
# We need these names because weather and holiday data
# must eventually be connected to the correct destination.

print("Number of destinations:", destination_df["Destination"].nunique())

print("\nDestinations:")
print(destination_df["Destination"].tolist())

Number of destinations: 92

Destinations:
['Goa', 'Ladakh (Leh)', 'Jaipur', 'Varanasi (Kashi)', 'Agra', 'Udaipur', 'Kerala Backwaters (Alappuzha & Kumarakom)', 'Coorg (Kodagu)', 'Ziro Valley', 'Mawlynnong & Nearby', 'Shimla', 'Dharamshala & Mcleod Ganj', 'Manali', 'Tawang', 'Khajjiar', 'Spiti Valley', 'Nubra Valley', 'Pangong Lake', 'Ooty', 'Munnar', 'Kanyakumari', 'Rann Of Kutch', 'Dhanushkodi', 'Chitrakote Falls', 'Gokarna', 'Pondicherry', 'Hampi', 'Majuli Island', 'Araku Valley', 'Varkala', 'Havelock Island', 'Neil Island', 'Shillong', 'Cherrapunji', 'Dawki', 'Krang Suri Falls', 'Rishikesh', 'Haridwar', 'Amritsar', 'Kanatal', 'Binsar', 'Munsiyari', 'Chakrata', 'Kanha National Park', 'Ranthambore National Park', 'Bandhavgarh National Park', 'Tadoba Andhari Tiger Reserve', 'Lakshadweep Circuit', 'Kalpeni Island', 'Jaisalmer', 'Kutch Interior Circuit', 'Wayanad', 'Tamhini Ghat & Mulshi', 'Mahabaleshwar & Panchgani', 'Chikmagalur', 'Amboli Ghat', 'Agumbe', 'Kudremukh', 'Port Blair', 'Da

In [36]:
# Define the expected file paths for the two external datasets.

weather_path = "../data/raw/weather_data.csv"
holiday_path = "../data/raw/holidays.csv"


# Try to load the weather dataset.
# If it does not exist yet, we will simply continue and
# create it during the data collection stage.

try:
    weather_df = pd.read_csv(weather_path)
    print("Weather dataset found.")
    print("Shape:", weather_df.shape)

except FileNotFoundError:
    print("Weather dataset not found yet.")


# Try to load the holiday dataset.

try:
    holiday_df = pd.read_csv(holiday_path)
    print("\nHoliday dataset found.")
    print("Shape:", holiday_df.shape)

except FileNotFoundError:
    print("Holiday dataset not found yet.")

Weather dataset not found yet.
Holiday dataset not found yet.


In [37]:
# Create a list of unique destinations.
# These are the locations for which we need weather information.

weather_destinations = destination_df[
    ["Destination", "State", "Best_Season"]
].drop_duplicates()

# Reset the index to make the table easier to work with.
weather_destinations = weather_destinations.reset_index(drop=True)

# Display the destinations that require weather data.
weather_destinations

,Destination,State,Best_Season
0,Goa,Goa,Nov-Feb
1,Ladakh (Leh),Ladakh,Mar-Jun
2,Jaipur,Rajasthan,Oct-Mar
3,Varanasi (Kashi),Uttar Pradesh,Oct-Mar
4,Agra,Uttar Pradesh,Oct-Mar
...,...,...,...
87,Gulmarg,Jammu & Kashmir,Mar-Jun
88,Auli,Uttarakhand,Mar-Jun
89,Tiruvannamalai,Tamil Nadu,Oct-Mar
90,Amarkantak,Madhya Pradesh,Oct-Mar


In [38]:
# Count how many destinations belong to each best-season category.
#
# This helps us understand the distribution of destinations
# before collecting seasonal weather information.

season_distribution = (
    destination_df["Best_Season"]
    .value_counts()
)

print(season_distribution)

Best_Season
Oct-Mar    46
Mar-Jun    30
Nov-Feb    16
Name: count, dtype: int64


In [39]:
# Create a separate table containing the destinations
# for which we need weather information.
#
# We keep the State because it helps identify the
# geographical location correctly.

weather_destinations = destination_df[
    ["Destination", "State", "Best_Season"]
].drop_duplicates()

# Reset the index after removing duplicates.
weather_destinations = weather_destinations.reset_index(drop=True)

# Display the first few destinations.
weather_destinations.head(10)

,Destination,State,Best_Season
0,Goa,Goa,Nov-Feb
1,Ladakh (Leh),Ladakh,Mar-Jun
2,Jaipur,Rajasthan,Oct-Mar
3,Varanasi (Kashi),Uttar Pradesh,Oct-Mar
4,Agra,Uttar Pradesh,Oct-Mar
5,Udaipur,Rajasthan,Oct-Mar
6,Kerala Backwaters (Alappuzha & Kumarakom),Kerala,Oct-Mar
7,Coorg (Kodagu),Karnataka,Mar-Jun
8,Ziro Valley,Arunachal Pradesh,Mar-Jun
9,Mawlynnong & Nearby,Meghalaya,Oct-Mar


In [40]:
# Define the columns that our collected weather data
# should eventually contain.
#
# These columns are designed for later feature engineering
# and recommendation-model integration.

weather_columns = [
    "Destination",
    "State",
    "Season",
    "Average_Temperature",
    "Average_Rainfall",
    "Average_Humidity",
    "Weather_Suitability"
]

# Display the planned schema.
print("Weather dataset columns:")
print(weather_columns)

Weather dataset columns:
['Destination', 'State', 'Season', 'Average_Temperature', 'Average_Rainfall', 'Average_Humidity', 'Weather_Suitability']


In [41]:
# Load environment variables from the project's .env file.
# This allows us to keep API keys outside the notebook.

import os
from dotenv import load_dotenv

# Load variables from .env.
load_dotenv()

# Read the weather API key.
weather_api_key = os.getenv("WEATHER_API_KEY")

# Check whether the key is available.
# We do NOT print the actual key for security reasons.

if weather_api_key:
    print("Weather API key found.")
else:
    print("Weather API key not found.")

Weather API key found.


In [42]:
# Import the requests library.
# It allows Python to send HTTP requests to the weather API.

import requests


# Select one destination from our dataset for testing.
# We use the first destination only to verify the API connection.

test_destination = weather_destinations.iloc[0]

print("Testing destination:", test_destination["Destination"])
print("State:", test_destination["State"])


# Build the API request.
# The exact URL/parameters depend on the weather API
# configured in your WEATHER_API_KEY.

weather_url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": test_destination["Destination"],
    "appid": weather_api_key,
    "units": "metric"
}


# Send the request to the weather API.

response = requests.get(
    weather_url,
    params=params,
    timeout=10
)


# Check the HTTP response status.

print("Status code:", response.status_code)

Testing destination: Goa
State: Goa
Status code: 200


In [43]:
# Convert the API response into JSON format.
#
# JSON makes it easier to access individual weather fields.

weather_response = response.json()


# Display the response so we can understand
# exactly what information the API provides.

weather_response

{'coord': {'lon': 74.0833, 'lat': 15.3333},
 'weather': [{'id': 500,
   'main': 'Rain',
   'description': 'light rain',
   'icon': '10d'}],
 'base': 'stations',
 'main': {'temp': 28.21,
  'feels_like': 32.46,
  'temp_min': 28.21,
  'temp_max': 28.21,
  'pressure': 1012,
  'humidity': 79,
  'sea_level': 1012,
  'grnd_level': 999},
 'visibility': 10000,
 'wind': {'speed': 3.54, 'deg': 256, 'gust': 5.33},
 'rain': {'1h': 0.64},
 'clouds': {'all': 93},
 'dt': 1787460919,
 'sys': {'type': 1,
  'id': 9233,
  'country': 'IN',
  'sunrise': 1787446215,
  'sunset': 1787491378},
 'timezone': 19800,
 'id': 1271157,
 'name': 'Goa',
 'cod': 200}

In [44]:
def get_weather(destination):
    """
    Fetch current weather data for a destination.
    """

    weather_url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": destination,
        "appid": weather_api_key,
        "units": "metric"
    }

    try:
        response = requests.get(
            weather_url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        weather_data = {
            "Destination": destination,
            "Temperature": data["main"]["temp"],
            "Feels_Like": data["main"]["feels_like"],
            "Humidity": data["main"]["humidity"],
            "Pressure": data["main"]["pressure"],
            "Weather": data["weather"][0]["main"],
            "Weather_Description": data["weather"][0]["description"],
            "Wind_Speed": data["wind"]["speed"],
            "Cloudiness": data["clouds"]["all"]
        }

        return weather_data

    except requests.exceptions.RequestException as e:
        print(f"Error fetching weather for {destination}: {e}")
        return None

In [45]:
goa_weather = get_weather("Goa")

goa_weather

{'Destination': 'Goa',
 'Temperature': 28.21,
 'Feels_Like': 32.46,
 'Humidity': 79,
 'Pressure': 1012,
 'Weather': 'Rain',
 'Weather_Description': 'light rain',
 'Wind_Speed': 3.54,
 'Cloudiness': 93}

In [46]:
# Display the number of destinations we need weather data for
print("Number of destinations:", len(weather_destinations))

# Display the first few destinations to verify the list
print("\nFirst few destinations:")
print(weather_destinations[:10])


Number of destinations: 92

First few destinations:
                                 Destination              State Best_Season
0                                        Goa                Goa     Nov-Feb
1                               Ladakh (Leh)             Ladakh     Mar-Jun
2                                     Jaipur          Rajasthan     Oct-Mar
3                           Varanasi (Kashi)      Uttar Pradesh     Oct-Mar
4                                       Agra      Uttar Pradesh     Oct-Mar
5                                    Udaipur          Rajasthan     Oct-Mar
6  Kerala Backwaters (Alappuzha & Kumarakom)             Kerala     Oct-Mar
7                             Coorg (Kodagu)          Karnataka     Mar-Jun
8                                Ziro Valley  Arunachal Pradesh     Mar-Jun
9                        Mawlynnong & Nearby          Meghalaya     Oct-Mar


In [47]:
# Create an empty list to store weather information
weather_results = []

# Keep track of destinations for which the API request fails
weather_failed = []

# Loop through every destination in our destination dataset
for destination in weather_destinations["Destination"]:


    # Call the weather API using our reusable function
    weather_data = get_weather(destination)

    # Check whether the API returned valid weather data
    if weather_data is not None:

        # Add the successful result to our list
        weather_results.append(weather_data)

    else:

        # Store the destination name if the API request failed
        weather_failed.append(destination)

# Convert the successful weather results into a DataFrame
weather_df = pd.DataFrame(weather_results)

# Display the basic results
print("Weather data collected successfully:", len(weather_df))
print("Weather data failed:", len(weather_failed))

Error fetching weather for Ladakh (Leh): 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Ladakh+%28Leh%29&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Varanasi (Kashi): 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Varanasi+%28Kashi%29&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Kerala Backwaters (Alappuzha & Kumarakom): 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Kerala+Backwaters+%28Alappuzha+%26+Kumarakom%29&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Coorg (Kodagu): 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Coorg+%28Kodagu%29&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Ziro Valley: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Ziro+Valley&a

In [48]:
weather_df

,Destination,Temperature,Feels_Like,Humidity,Pressure,Weather,Weather_Description,Wind_Speed,Cloudiness
0,Goa,28.21,32.46,79,1012,Rain,light rain,3.54,93
1,Jaipur,30.62,33.61,58,1006,Clouds,broken clouds,3.09,55
2,Agra,33.59,37.51,50,1004,Clouds,broken clouds,2.04,58
3,Udaipur,27.50,29.42,67,1008,Clouds,broken clouds,3.95,51
4,Shimla,20.39,20.78,88,1008,Clear,clear sky,0.00,1
5,Manali,33.05,38.26,56,1008,Clouds,overcast clouds,5.57,100
6,Tawang,20.73,20.45,61,1011,Rain,light rain,3.26,98
7,Khajjiar,23.30,23.54,71,1008,Clear,clear sky,2.03,9
8,Ooty,16.82,16.28,66,1014,Clouds,overcast clouds,1.78,100
9,Munnar,20.00,19.96,73,1015,Clouds,overcast clouds,3.20,95


In [49]:
# Display the destinations for which weather data could not be fetched
print("Failed destinations:")
print(weather_failed)

Failed destinations:
['Ladakh (Leh)', 'Varanasi (Kashi)', 'Kerala Backwaters (Alappuzha & Kumarakom)', 'Coorg (Kodagu)', 'Ziro Valley', 'Mawlynnong & Nearby', 'Dharamshala & Mcleod Ganj', 'Spiti Valley', 'Nubra Valley', 'Pangong Lake', 'Rann Of Kutch', 'Dhanushkodi', 'Chitrakote Falls', 'Majuli Island', 'Araku Valley', 'Havelock Island', 'Neil Island', 'Dawki', 'Krang Suri Falls', 'Kanatal', 'Munsiyari', 'Kanha National Park', 'Ranthambore National Park', 'Bandhavgarh National Park', 'Tadoba Andhari Tiger Reserve', 'Lakshadweep Circuit', 'Kalpeni Island', 'Kutch Interior Circuit', 'Wayanad', 'Tamhini Ghat & Mulshi', 'Mahabaleshwar & Panchgani', 'Amboli Ghat', 'Kudremukh', 'North Sikkim', 'Meghalaya Circuit', 'Kaziranga', 'Khajuraho & Panna', 'Almora & Kasar Devi', 'Jibhi & Tirthan', 'Chitkul & Baspa', 'Miyar Valley', 'Keoladeo National Park', 'Ranganathittu Bird Sanctuary', 'Nal Sarovar', 'Great Rann Of Kutch', 'Maravanthe', 'Astaranga & Ramachandi', 'Unakoti', 'Hidden Goa Coves', 'Ban

In [50]:
# Create a mapping for destinations whose original names
# are not directly recognized by the OpenWeather city search.
#
# We keep the original destination name unchanged in our dataset.
# The mapped value is only used when making the weather API request.

weather_query_mapping = {

    # Ladakh region
    "Ladakh (Leh)": "Leh",
    "Nubra Valley": "Diskit",
    "Pangong Lake": "Leh",
    "Spiti Valley": "Kaza",
    "Miyar Valley": "Keylong",

    # Uttar Pradesh / North India
    "Varanasi (Kashi)": "Varanasi",
    "Varanasi Ganga Ghats": "Varanasi",
    "Dharamshala & Mcleod Ganj": "Dharamshala",
    "Kanatal": "Kanatal",
    "Munsiyari": "Munsiyari",
    "Almora & Kasar Devi": "Almora",
    "Jibhi & Tirthan": "Jibhi",
    "Chitkul & Baspa": "Chitkul",

    # Kerala
    "Kerala Backwaters (Alappuzha & Kumarakom)": "Alappuzha",
    "Wayanad": "Kalpetta",
    "Malanad–Malabar Belt": "Kozhikode",
    "Maravanthe": "Kundapura",

    # Karnataka
    "Coorg (Kodagu)": "Madikeri",
    "Kudremukh": "Kalasa",
    "Amboli Ghat": "Amboli",
    "Tamhini Ghat & Mulshi": "Mulshi",
    "Mahabaleshwar & Panchgani": "Mahabaleshwar",
    "Hampi & Pattadakal": "Hampi",
    "Ranganathittu Bird Sanctuary": "Mysore",

    # Northeast India
    "Ziro Valley": "Ziro",
    "Mawlynnong & Nearby": "Shillong",
    "Dawki": "Dawki",
    "Krang Suri Falls": "Jowai",
    "Majuli Island": "Jorhat",
    "North Sikkim": "Mangan",
    "Meghalaya Circuit": "Shillong",
    "Kaziranga": "Kaziranga",
    "Unakoti": "Kailashahar",

    # Gujarat / Rajasthan
    "Rann Of Kutch": "Bhuj",
    "Great Rann Of Kutch": "Bhuj",
    "Kutch Interior Circuit": "Bhuj",
    "Nal Sarovar": "Ahmedabad",

    # Tamil Nadu
    "Dhanushkodi": "Rameswaram",

    # Chhattisgarh
    "Chitrakote Falls": "Jagdalpur",

    # Andhra Pradesh
    "Araku Valley": "Araku Valley",

    # Andaman and Nicobar
    "Havelock Island": "Port Blair",
    "Neil Island": "Port Blair",

    # Uttarakhand
    "Rishikesh & Haridwar Route": "Rishikesh",
    "Haridwar–Char Dham Trail": "Haridwar",

    # Wildlife destinations
    "Kanha National Park": "Kanha",
    "Ranthambore National Park": "Sawai Madhopur",
    "Bandhavgarh National Park": "Umaria",
    "Tadoba Andhari Tiger Reserve": "Chandrapur",
    "Keoladeo National Park": "Bharatpur",

    # Lakshadweep
    "Lakshadweep Circuit": "Kavaratti",
    "Kalpeni Island": "Kavaratti",
    "Bangaram & Kadmat": "Kavaratti",

    # Madhya Pradesh
    "Khajuraho & Panna": "Khajuraho",

    # Goa
    "Hidden Goa Coves": "Goa",

    # Odisha
    "Astaranga & Ramachandi": "Puri",

    # Andhra / Telangana heritage
    "Krishna Heritage Circuit": "Vijayawada"
}

# Display how many failed destinations have been mapped
print("Number of mapped destinations:", len(weather_query_mapping))

Number of mapped destinations: 56


In [51]:
# Check which failed destinations are still missing
# from our weather query mapping.

unmapped_destinations = [
    destination
    for destination in weather_failed
    if destination not in weather_query_mapping
]

print("Failed destinations:", len(weather_failed))
print("Mapped destinations:", len(weather_query_mapping))
print("Still unmapped:", len(unmapped_destinations))

print("\nStill unmapped destinations:")
print(unmapped_destinations)

Failed destinations: 56
Mapped destinations: 56
Still unmapped: 0

Still unmapped destinations:
[]


In [52]:
# Create a list to store the weather data for the destinations
# that previously failed.

retry_weather_results = []

# Loop through every destination that previously failed
for destination in weather_failed:

    # Get the alternative city/location name from our mapping.
    # If a mapping does not exist, use the original destination name.
    weather_query = weather_query_mapping.get(destination, destination)

    # Fetch weather using the mapped location name
    weather_data = get_weather(weather_query)

    # Check whether the API returned valid data
    if weather_data is not None:

        # Replace the API query name with the original
        # destination name used in our tourism dataset.
        weather_data["Destination"] = destination

        # Store the successful result
        retry_weather_results.append(weather_data)

# Convert the retry results into a DataFrame
retry_weather_df = pd.DataFrame(retry_weather_results)

# Display the number of successfully recovered destinations
print("Previously failed destinations recovered:",
      len(retry_weather_df))

# Display the recovered weather data
retry_weather_df.head(47)

Error fetching weather for Dharamshala: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Dharamshala&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Diskit: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Diskit&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Araku Valley: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Araku+Valley&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Dawki: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Dawki&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Kanatal: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Kanatal&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
Error fetching weather for Munsiyari: 404 Client Error: Not Found for url

,Destination,Temperature,Feels_Like,Humidity,Pressure,Weather,Weather_Description,Wind_Speed,Cloudiness
0,Ladakh (Leh),14.83,13.49,43,1010,Clear,clear sky,3.02,7
1,Varanasi (Kashi),30.05,37.05,84,1003,Clouds,overcast clouds,0.00,100
2,Kerala Backwaters (Alappuzha & Kumarakom),27.38,30.04,75,1014,Clouds,broken clouds,3.56,58
3,Coorg (Kodagu),22.02,22.47,84,1015,Rain,light rain,2.59,100
4,Ziro Valley,22.55,22.90,78,1008,Rain,light rain,0.52,100
5,Mawlynnong & Nearby,23.02,23.70,89,1009,Rain,light rain,0.00,100
6,Spiti Valley,27.87,26.77,24,1008,Clear,clear sky,2.39,0
7,Pangong Lake,14.83,13.49,43,1010,Clear,clear sky,3.02,7
8,Rann Of Kutch,30.40,33.61,60,1006,Clouds,overcast clouds,7.69,88
9,Dhanushkodi,29.12,33.38,72,1012,Clouds,broken clouds,5.74,84


In [53]:
# Find the destinations that still could not be retrieved
# after using the alternative city/location names.

retry_successful_destinations = set(
    retry_weather_df["Destination"]
)

still_failed = [
    destination
    for destination in weather_failed
    if destination not in retry_successful_destinations
]

print("Still failed:", len(still_failed))

print("\nDestinations still needing attention:")
print(still_failed)

Still failed: 9

Destinations still needing attention:
['Dharamshala & Mcleod Ganj', 'Nubra Valley', 'Araku Valley', 'Dawki', 'Kanatal', 'Munsiyari', 'Kaziranga', 'Chitkul & Baspa', 'Miyar Valley']


In [54]:
# Latitude and longitude for destinations that could not be
# resolved reliably using the OpenWeather city-name search.
#
# Coordinates allow us to request weather directly for a location
# instead of depending on the city-name lookup.

weather_coordinates = {

    "Dharamshala & Mcleod Ganj": (32.2190, 76.3234),
    "Nubra Valley": (34.6500, 77.5500),
    "Araku Valley": (18.3273, 82.8730),
    "Dawki": (25.1920, 92.0240),
    "Kanatal": (30.3400, 78.3500),
    "Munsiyari": (30.0687, 80.2385),
    "Kaziranga": (26.5775, 93.1711),
    "Chitkul & Baspa": (31.3530, 78.4370),
    "Miyar Valley": (32.8500, 76.8500)
}

# Check that we have coordinates for all remaining destinations
print("Coordinates available:", len(weather_coordinates))

Coordinates available: 9


In [55]:
def get_weather_by_coordinates(destination, latitude, longitude):
    """
    Fetch current weather using latitude and longitude.

    This function is used when OpenWeather cannot reliably
    identify a destination using its city name.
    """

    # OpenWeather current weather API endpoint
    weather_url = "https://api.openweathermap.org/data/2.5/weather"

    # Parameters sent to the API
    params = {
        "lat": latitude,
        "lon": longitude,
        "appid": weather_api_key,
        "units": "metric"
    }

    try:
        # Send the request to OpenWeather
        response = requests.get(
            weather_url,
            params=params,
            timeout=10
        )

        # Raise an error if the API request failed
        response.raise_for_status()

        # Convert the API response into JSON
        data = response.json()

        # Extract only the weather features required
        # for our Travel Agent project
        weather_data = {
            "Destination": destination,
            "Temperature": data["main"]["temp"],
            "Feels_Like": data["main"]["feels_like"],
            "Humidity": data["main"]["humidity"],
            "Pressure": data["main"]["pressure"],
            "Weather": data["weather"][0]["main"],
            "Weather_Description": data["weather"][0]["description"],
            "Wind_Speed": data["wind"]["speed"],
            "Cloudiness": data["clouds"]["all"]
        }

        return weather_data

    except requests.exceptions.RequestException as e:

        # Print the error but allow the remaining destinations
        # to continue processing
        print(f"Error fetching weather for {destination}: {e}")

        return None

In [56]:
# Create a list to store weather results obtained
# using latitude and longitude.
coordinate_weather_results = []

# Loop through the remaining failed destinations
for destination in still_failed:

    # Get the coordinates for the current destination
    latitude, longitude = weather_coordinates[destination]

    # Fetch weather using the coordinates
    weather_data = get_weather_by_coordinates(
        destination,
        latitude,
        longitude
    )

    # Store the result if the API request was successful
    if weather_data is not None:
        coordinate_weather_results.append(weather_data)

# Convert the results into a DataFrame
coordinate_weather_df = pd.DataFrame(
    coordinate_weather_results
)

# Display the number of successful requests
print(
    "Coordinate-based weather records:",
    len(coordinate_weather_df)
)

# Display the collected data
coordinate_weather_df

Coordinate-based weather records: 9


,Destination,Temperature,Feels_Like,Humidity,Pressure,Weather,Weather_Description,Wind_Speed,Cloudiness
0,Dharamshala & Mcleod Ganj,23.24,23.55,74,1009,Rain,light rain,2.70,16
1,Nubra Valley,13.00,11.22,33,1007,Clear,clear sky,1.28,3
2,Araku Valley,21.82,22.38,89,1007,Clouds,overcast clouds,4.68,100
3,Dawki,30.72,37.72,84,1007,Rain,light rain,1.67,100
4,Kanatal,22.97,23.65,89,1008,Rain,moderate rain,1.96,14
5,Munsiyari,19.54,19.38,70,1012,Clouds,overcast clouds,1.54,100
6,Kaziranga,32.37,39.37,77,1005,Clouds,overcast clouds,0.49,100
7,Chitkul & Baspa,10.02,8.85,68,1014,Rain,light rain,3.32,62
8,Miyar Valley,14.19,13.15,57,1014,Clouds,few clouds,2.72,12


In [57]:
# Combine the weather data collected from the three approaches:
#
# 1. weather_df
#    → Destinations successfully fetched using the original name
#
# 2. retry_weather_df
#    → Destinations recovered using alternative city/location names
#
# 3. coordinate_weather_df
#    → Destinations recovered using latitude and longitude

weather_df = pd.concat(
    [
        weather_df,
        retry_weather_df,
        coordinate_weather_df
    ],
    ignore_index=True
)

# Display the total number of weather records
print("Total weather records:", len(weather_df))

# Display the first few rows of the final weather dataset
weather_df.head()

Total weather records: 92


,Destination,Temperature,Feels_Like,Humidity,Pressure,Weather,Weather_Description,Wind_Speed,Cloudiness
0,Goa,28.21,32.46,79,1012,Rain,light rain,3.54,93
1,Jaipur,30.62,33.61,58,1006,Clouds,broken clouds,3.09,55
2,Agra,33.59,37.51,50,1004,Clouds,broken clouds,2.04,58
3,Udaipur,27.50,29.42,67,1008,Clouds,broken clouds,3.95,51
4,Shimla,20.39,20.78,88,1008,Clear,clear sky,0.00,1


In [58]:
# Check whether any destination appears more than once
# in the final weather dataset.

duplicate_count = weather_df["Destination"].duplicated().sum()

print("Duplicate destinations:", duplicate_count)

Duplicate destinations: 0


In [60]:
# Display duplicate destination names if any exist.
# This will return an empty DataFrame when there are no duplicates.

weather_df[
    weather_df["Destination"].duplicated(keep=False)
].sort_values("Destination")

,Destination,Temperature,Feels_Like,Humidity,Pressure,Weather,Weather_Description,Wind_Speed,Cloudiness


In [62]:
# Check every column for missing values.
#
# Missing values are important to identify before we merge
# the weather data with our main destination dataset.

weather_df.isnull().sum()

Destination            0
Temperature            0
Feels_Like             0
Humidity               0
Pressure               0
Weather                0
Weather_Description    0
Wind_Speed             0
Cloudiness             0
dtype: int64

In [61]:
# Check the data type of every weather feature.
#
# This helps ensure that numerical features such as temperature,
# humidity, pressure, and wind speed are stored as numbers.

weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Destination          92 non-null     str    
 1   Temperature          92 non-null     float64
 2   Feels_Like           92 non-null     float64
 3   Humidity             92 non-null     int64  
 4   Pressure             92 non-null     int64  
 5   Weather              92 non-null     str    
 6   Weather_Description  92 non-null     str    
 7   Wind_Speed           92 non-null     float64
 8   Cloudiness           92 non-null     int64  
dtypes: float64(3), int64(3), str(3)
memory usage: 6.6 KB


In [63]:
# Define the path where we will save the cleaned weather dataset.
# This keeps all processed datasets inside the data/cleaned folder.

weather_output_path = "../data/cleaned/weather_data.csv"

# Save the weather DataFrame as a CSV file.
# index=False prevents pandas from saving the DataFrame index
# as an unnecessary column in the CSV file.

weather_df.to_csv(
    weather_output_path,
    index=False
)

# Confirm that the file was saved successfully.
print("Weather dataset saved successfully!")
print("File path:", weather_output_path)

Weather dataset saved successfully!
File path: ../data/cleaned/weather_data.csv


In [64]:
# Load the saved weather dataset again.
# This verifies that the CSV file was written correctly.

weather_check = pd.read_csv(
    weather_output_path
)

# Display the shape of the loaded dataset.
# Expected result: 92 rows and 10 columns.

print("Dataset shape:", weather_check.shape)

# Display the first five rows to verify the data.
weather_check.head()

Dataset shape: (92, 9)


,Destination,Temperature,Feels_Like,Humidity,Pressure,Weather,Weather_Description,Wind_Speed,Cloudiness
0,Goa,28.21,32.46,79,1012,Rain,light rain,3.54,93
1,Jaipur,30.62,33.61,58,1006,Clouds,broken clouds,3.09,55
2,Agra,33.59,37.51,50,1004,Clouds,broken clouds,2.04,58
3,Udaipur,27.50,29.42,67,1008,Clouds,broken clouds,3.95,51
4,Shimla,20.39,20.78,88,1008,Clear,clear sky,0.00,1


In [66]:
# Check the number of destinations in our main destination DataFrame.
# In our project, the cleaned destination features are stored in destination_df.

print("Destinations in destination_df:",
      len(destination_df))

# Check the number of destinations in our weather dataset.

print("Destinations in weather_df:",
      len(weather_df))

# Display a few destination names from both DataFrames
# to make sure they use the same naming format.

print("\nDestination examples from destination_df:")
print(destination_df["Destination"].head(10).tolist())

print("\nDestination examples from weather_df:")
print(weather_df["Destination"].head(10).tolist())

Destinations in destination_df: 92
Destinations in weather_df: 92

Destination examples from destination_df:
['Goa', 'Ladakh (Leh)', 'Jaipur', 'Varanasi (Kashi)', 'Agra', 'Udaipur', 'Kerala Backwaters (Alappuzha & Kumarakom)', 'Coorg (Kodagu)', 'Ziro Valley', 'Mawlynnong & Nearby']

Destination examples from weather_df:
['Goa', 'Jaipur', 'Agra', 'Udaipur', 'Shimla', 'Manali', 'Tawang', 'Khajjiar', 'Ooty', 'Munnar']


In [67]:
# Find destinations that exist in destination_df
# but are missing from the weather dataset.

missing_weather_destinations = set(
    destination_df["Destination"]
) - set(
    weather_df["Destination"]
)

# Find destinations that exist in weather_df
# but are not present in destination_df.

extra_weather_destinations = set(
    weather_df["Destination"]
) - set(
    destination_df["Destination"]
)

print("Destinations missing weather data:",
      len(missing_weather_destinations))

print("Extra weather destinations:",
      len(extra_weather_destinations))

print("\nMissing weather destinations:")
print(missing_weather_destinations)

print("\nExtra weather destinations:")
print(extra_weather_destinations)

Destinations missing weather data: 0
Extra weather destinations: 0

Missing weather destinations:
set()

Extra weather destinations:
set()


In [68]:
# Merge the destination features with the weather data.
#
# "Destination" is the common column between both DataFrames.
#
# We use a LEFT JOIN so that every destination from destination_df
# is preserved even if weather data is missing for any destination.

destination_weather = destination_df.merge(
    weather_df,
    on="Destination",
    how="left"
)

# Display the shape of the newly integrated dataset.

print("Integrated dataset shape:",
      destination_weather.shape)

# Display the first five rows of the integrated dataset.

destination_weather.head()

Integrated dataset shape: (92, 55)


,Destination,State,Category,Best_Season,Average_Budget,Recommended_Trip_Duration,Nearest_Airport,Popularity,Family_Friendly,Adventure_Score,...,Hotel_Data_Available,Supporting_Data_Available,Temperature,Feels_Like,Humidity,Pressure,Weather,Weather_Description,Wind_Speed,Cloudiness
0,Goa,Goa,Beach,Nov-Feb,30000,4,Goa Airport,High,1,6,...,1,1,28.21,32.46,79,1012,Rain,light rain,3.54,93
1,Ladakh (Leh),Ladakh,Mountain,Mar-Jun,35000,5,Ladakh Airport,High,1,9,...,0,0,14.83,13.49,43,1010,Clear,clear sky,3.02,7
2,Jaipur,Rajasthan,Heritage,Oct-Mar,22000,3,Jaipur Airport,High,1,3,...,1,1,30.62,33.61,58,1006,Clouds,broken clouds,3.09,55
3,Varanasi (Kashi),Uttar Pradesh,Spiritual,Oct-Mar,18000,2,Varanasi Airport,High,1,2,...,1,1,30.05,37.05,84,1003,Clouds,overcast clouds,0.00,100
4,Agra,Uttar Pradesh,Heritage,Oct-Mar,22000,3,Agra Airport,High,1,3,...,0,1,33.59,37.51,50,1004,Clouds,broken clouds,2.04,58


In [69]:
# Check the number of rows and columns in the final integrated dataset.
# We expect 92 destinations and 55 features.

print("Integrated dataset shape:", destination_weather.shape)


# Check whether any duplicate destinations were created during the merge.
# Each destination should appear only once.

print(
    "Duplicate destinations:",
    destination_weather["Destination"].duplicated().sum()
)


# Check for missing values in every column.
# This is especially important for the newly added weather columns.

missing_values = destination_weather.isnull().sum()

print("\nColumns with missing values:")
print(missing_values[missing_values > 0])

Integrated dataset shape: (92, 55)
Duplicate destinations: 0

Columns with missing values:
Hotel_City              83
Average_Hotel_Rating    83
Average_Hotel_Price     83
Minimum_Hotel_Price     83
Maximum_Hotel_Price     83
Number_of_Hotels        83
Average_Flight_Cost     81
Minimum_Flight_Cost     81
Maximum_Flight_Cost     81
Number_of_Routes        81
dtype: int64


In [70]:
# List the weather-related columns that were added from the OpenWeather API.

weather_columns = [
    "Temperature",
    "Feels_Like",
    "Humidity",
    "Pressure",
    "Weather",
    "Weather_Description",
    "Wind_Speed",
    "Cloudiness"
]


# Check missing values only in the weather columns.

print("Missing values in weather features:")
print(destination_weather[weather_columns].isnull().sum())

Missing values in weather features:
Temperature            0
Feels_Like             0
Humidity               0
Pressure               0
Weather                0
Weather_Description    0
Wind_Speed             0
Cloudiness             0
dtype: int64


In [71]:
# Define the output path for our integrated destination dataset.
# This dataset contains the original destination features
# plus the weather information obtained from OpenWeather.

integrated_output_path = "../data/cleaned/destination_weather_features.csv"


# Save the integrated DataFrame as a CSV file.
# index=False prevents pandas from saving the DataFrame index.

destination_weather.to_csv(
    integrated_output_path,
    index=False
)


# Confirm that the file was saved successfully.

print("Integrated dataset saved successfully!")
print("File path:", integrated_output_path)

Integrated dataset saved successfully!
File path: ../data/cleaned/destination_weather_features.csv


### Holidays Data Collection

In [75]:
# Define the year for which we want holiday information.
# We are using 2026 for our Travel Agent project.

holiday_year = 2026


# Define the country code for India.
# "IN" represents India in the ISO country-code standard.

country_code = "IN"


# Build the Nager.Date API URL.
# The current Nager.Date project examples use nagerholidays.com.

holiday_url = (
    f"https://nagerholidays.com/api/v3/publicholidays/"
    f"{holiday_year}/{country_code}"
)


# Send a GET request to the holiday API.

holiday_response = requests.get(
    holiday_url,
    timeout=10
)


# Display the HTTP response status code.
# 200 means the request was successful.

print("Status code:", holiday_response.status_code)

Status code: 204


In [74]:
# Check the HTTP status code returned by the holiday API.

print("Status code:", holiday_response.status_code)


# Check whether the API actually returned any content.
# HTTP 204 means that the server returned no content.

if holiday_response.status_code == 204:

    print("The API returned 204 - No Content.")
    print("No JSON data is available in this response.")

else:

    # Convert the response to JSON only when content exists.
    holiday_data = holiday_response.json()

    # Display the first holiday record so that
    # we can inspect the API response structure.

    print("Number of holidays:", len(holiday_data))
    print("\nFirst holiday:")
    print(holiday_data[0])

Status code: 204
The API returned 204 - No Content.
No JSON data is available in this response.


In [76]:
# ---------------------------------------------------------
# HOLIDAY DATA INTEGRATION
# ---------------------------------------------------------

# We are using the Government of India Department of Posts
# All India Holiday Calendar for the year 2026.
#
# Source:
# https://www.indiapost.gov.in/holidays-list
#
# The source provides the official All India holiday dates
# for 2026.


# Create a list containing the official All India holidays
# published for 2026.

holiday_records = [
    ("Republic Day", "26-January-2026"),
    ("Id-ul-Fitr", "21-March-2026"),
    ("Mahavir Jayanti", "31-March-2026"),
    ("Good Friday", "03-April-2026"),
    ("Buddha Purnima", "01-May-2026"),
    ("Id-ul-Zuha", "27-May-2026"),
    ("Muharram", "26-June-2026"),
    ("Independence Day", "15-August-2026"),
    ("Prophet Mohammad's Birthday (Id-e-Milad)", "26-August-2026"),
    ("Mahatma Gandhi's Birthday", "02-October-2026"),
    ("Dussehra (Vijay Dashmi)", "20-October-2026"),
    ("Diwali (Deepavali)", "08-November-2026"),
    ("Guru Nanak's Birthday", "24-November-2026"),
    ("Christmas Day", "25-December-2026")
]


# Convert the list into a DataFrame.
#
# This gives us a structured table that can be cleaned
# and transformed using pandas.

holiday_df = pd.DataFrame(
    holiday_records,
    columns=["Holiday_Name", "Date"]
)


# Convert the Date column from text into pandas datetime format.
#
# Datetime format is important because it allows us to
# extract month, weekday, year, etc.

holiday_df["Date"] = pd.to_datetime(
    holiday_df["Date"],
    format="%d-%B-%Y"
)


# Display the first few rows to verify the data.

holiday_df.head()

,Holiday_Name,Date
0,Republic Day,2026-01-26
1,Id-ul-Fitr,2026-03-21
2,Mahavir Jayanti,2026-03-31
3,Good Friday,2026-04-03
4,Buddha Purnima,2026-05-01


In [77]:
# Extract the year from the holiday date.
#
# Although all records currently belong to 2026,
# keeping the year as a feature makes the dataset easier
# to extend to future years.

holiday_df["Year"] = holiday_df["Date"].dt.year


# Extract the numeric month.
#
# Example:
# January = 1
# February = 2
# ...
# December = 12

holiday_df["Month"] = holiday_df["Date"].dt.month


# Extract the month name.
#
# This is easier for humans to understand when exploring
# the dataset.

holiday_df["Month_Name"] = holiday_df["Date"].dt.month_name()


# Extract the day of the week.

holiday_df["Day"] = holiday_df["Date"].dt.day_name()


# Identify whether the holiday falls on Saturday or Sunday.
#
# Monday = 0
# Tuesday = 1
# ...
# Saturday = 5
# Sunday = 6

holiday_df["Is_Weekend"] = (
    holiday_df["Date"].dt.dayofweek >= 5
)


# Display the transformed holiday dataset.

holiday_df

,Holiday_Name,Date,Year,Month,Month_Name,Day,Is_Weekend
0,Republic Day,2026-01-26,2026,1,January,Monday,False
1,Id-ul-Fitr,2026-03-21,2026,3,March,Saturday,True
2,Mahavir Jayanti,2026-03-31,2026,3,March,Tuesday,False
3,Good Friday,2026-04-03,2026,4,April,Friday,False
4,Buddha Purnima,2026-05-01,2026,5,May,Friday,False
5,Id-ul-Zuha,2026-05-27,2026,5,May,Wednesday,False
6,Muharram,2026-06-26,2026,6,June,Friday,False
7,Independence Day,2026-08-15,2026,8,August,Saturday,True
8,Prophet Mohammad's Birthday (Id-e-Milad),2026-08-26,2026,8,August,Wednesday,False
9,Mahatma Gandhi's Birthday,2026-10-02,2026,10,October,Friday,False


In [78]:
# Check the number of holiday records.

print("Number of holidays:", len(holiday_df))


# Check for duplicate holiday dates.

print(
    "Duplicate holiday dates:",
    holiday_df["Date"].duplicated().sum()
)


# Check for missing values in each column.

print("\nMissing values:")
print(holiday_df.isnull().sum())

Number of holidays: 14
Duplicate holiday dates: 0

Missing values:
Holiday_Name    0
Date            0
Year            0
Month           0
Month_Name      0
Day             0
Is_Weekend      0
dtype: int64


In [79]:
# Count how many All India holidays occur in each month.

holiday_month_counts = (
    holiday_df
    .groupby(["Month", "Month_Name"])
    .size()
    .reset_index(name="Holiday_Count")
)


# Sort the result by month number.

holiday_month_counts = holiday_month_counts.sort_values(
    "Month"
)


# Display the monthly holiday distribution.

holiday_month_counts

,Month,Month_Name,Holiday_Count
0,1,January,1
1,3,March,2
2,4,April,1
3,5,May,2
4,6,June,1
5,8,August,2
6,10,October,2
7,11,November,2
8,12,December,1


In [80]:
# Define the output path for the cleaned holiday dataset.

holiday_output_path = (
    "../data/cleaned/holiday_data.csv"
)


# Save the holiday DataFrame as a CSV file.
#
# index=False prevents pandas from saving the DataFrame
# index as an extra column.

holiday_df.to_csv(
    holiday_output_path,
    index=False
)


# Confirm that the file was saved successfully.

print("Holiday dataset saved successfully!")
print("File path:", holiday_output_path)
print("Shape:", holiday_df.shape)

Holiday dataset saved successfully!
File path: ../data/cleaned/holiday_data.csv
Shape: (14, 7)
